# LLM Wiki — A Self-Generating Wiki Grounded in Your Documents

This notebook builds a small **encyclopedia written by an LLM, but grounded in your own documents**. It runs **entirely locally**, except for the concept-discovery, distillation, page-generation, and answer calls, which use **Groq's `llama-3.3-70b-versatile`**.

It first parses whatever `.txt` / `.md` / `.pdf` files are in `data/` (the same loader as `rag_demo.ipynb` and `rag_graph_demo.ipynb`), discovers the key concepts they contain, distills those down to the most important ones, then asks Groq to write a wiki article for each concept **using only retrieved document excerpts as context** — skipping any concept the documents don't actually cover — and to suggest related concepts to crawl next, breadth-first, the same way a reader follows hyperlinks between wiki articles. The result is a small, browsable wiki that stays faithful to your source material, plus a graph of how its concepts connect.

Pipeline:
1. **Load** documents from `data/` (`.txt`, `.md`, `.pdf`)
2. **Split** each file into overlapping chunks and **embed** them locally, storing them in a vector DB for retrieval
3. **Discover** candidate wiki concepts by asking Groq to scan a sample of chunks
4. **Distill** the candidates down to the most important, central concepts
5. **Generate** wiki pages breadth-first: for each concept, retrieve the most relevant document chunks, skip it if none are relevant enough, otherwise ask Groq to write an article grounded in them (plus related-concept links to crawl next)
6. **Persist** each page as a Markdown file under `wiki/`, citing the source document(s) it was grounded in
7. **Build** a directed link graph (`networkx`) — nodes are wiki concepts, edges are `concept → linked concept`
8. **Visualize** the wiki as an interactive, force-directed graph (inline Plotly + a standalone draggable [pyvis](https://pyvis.readthedocs.io/) HTML file)
9. **Chunk + embed** each generated wiki page and store it in its own local vector DB
10. **Ask a question** → retrieve the most relevant wiki chunks, highlight their source pages in the graph
11. **Compile an answer** with Groq, rendered as right-to-left HTML (for Hebrew/RTL source material) the same way as the other notebooks

Only Groq is used as an external API (for concept discovery, distillation, page generation, and answer generation) — parsing, chunking, embeddings, both vector DBs, and both graph visualizations run locally.


## 0. Setup

This notebook expects to run under the `.venv` virtual environment in this project folder — select **Python (.venv)** as the kernel (top-right in VS Code / Jupyter) before running the cells below.

Dependencies are listed in `requirements.txt`; the cell below installs them into `.venv`. API keys are loaded from a local `.env` file via `python-dotenv` — copy `.env.example` to `.env` and fill in your Groq key (free tier at https://console.groq.com/keys):

```
GROQ_API_KEY=your-key-here
```

`.env` is never read by anything outside this notebook/process, so it keeps the key out of the notebook source itself. Drop `.txt`, `.md`, or `.pdf` files into the `data/` folder before running — the wiki is built entirely from whatever it finds there.

In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import os
import re
import json
import time
import glob
import html as html_lib
from collections import deque
from pathlib import Path

import pandas as pd
import networkx as nx
import plotly.io as pio
import plotly.graph_objects as go
from dotenv import load_dotenv
from IPython.display import HTML, display

import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
from pypdf import PdfReader
from pyvis.network import Network

load_dotenv()

# Plotly's own default renderer is "browser" (opens a new OS browser tab), which shows nothing
# inside the notebook itself — force the renderer VS Code's Jupyter extension knows how to
# display inline.
pio.renderers.default = "vscode"

DATA_DIR = Path("data")                    # source documents to build the wiki from
WIKI_DIR = Path("wiki")                     # generated Markdown pages
WIKI_GRAPH_DIR = Path("wiki_graph_db")      # persisted graph + rendered HTML views
MODEL_CACHE_DIR = Path(".model_cache")      # local cache so the embedding model isn't re-downloaded each run

CHUNK_SIZE = 500          # characters per chunk
CHUNK_OVERLAP = 80        # overlap between consecutive chunks
EMBED_MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"
GROQ_MODEL = "llama-3.3-70b-versatile"

MAX_CONCEPT_CHUNKS = 30      # number of document chunks scanned for candidate wiki concepts (keep the demo fast & inside free-tier rate limits)
MAX_SEED_CONCEPTS = 20       # candidate concepts distilled down to this many before crawling starts
MAX_PAGES = 30               # page budget for the wiki
MAX_LINKS_PER_PAGE = 5       # cap on related-concept links Groq returns per page, to keep the graph readable
TOP_K = 4                    # document chunks retrieved as grounding context per wiki page, and per question
TOP_K_NODES = 3              # number of seed pages matched per question, for the highlighted graph view
HOPS = 1                     # how many link hops to expand around the retrieved pages, for the highlighted graph view
MIN_RELEVANCE_DISTANCE = 0.5 # skip writing a page if even the closest document chunk is farther than this (cosine distance, lower = more relevant)

DATA_DIR.mkdir(exist_ok=True)
WIKI_DIR.mkdir(exist_ok=True)
WIKI_GRAPH_DIR.mkdir(exist_ok=True)

# Clear out wiki pages from a previous run before digesting the input files again — otherwise
# pages for concepts that don't recur this run (or that used to fail grounding) linger on disk.
for old_page in WIKI_DIR.glob("*.md"):
    old_page.unlink()


## 1. Load documents

Drop `.txt`, `.md`, or `.pdf` files into the `data/` folder, then run the cell below to load them. (Reuses the same loader as `rag_demo.ipynb` and `rag_graph_demo.ipynb`.)

In [ ]:
def load_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

text_paths = sorted(glob.glob(str(DATA_DIR / "*.txt"))) + sorted(glob.glob(str(DATA_DIR / "*.md")))
pdf_paths = sorted(glob.glob(str(DATA_DIR / "*.pdf")))

documents = {}
for path in text_paths:
    documents[Path(path).name] = Path(path).read_text(encoding="utf-8", errors="ignore")
for path in pdf_paths:
    documents[Path(path).name] = load_pdf(path)

assert documents, f"No .txt/.md/.pdf files found in {DATA_DIR}/ — add some documents and re-run this cell."
print(f"Loaded {len(documents)} files: {list(documents.keys())}")


## 2. Split into chunks and embed them

Same fixed-size sliding-window splitter as the other notebooks. Chunks are embedded locally and stored in a vector DB — this is what grounds every wiki page and answer in the actual document text, instead of the LLM's own general knowledge.

In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

records = []  # each: {id, text, source}
for source, text in documents.items():
    for i, chunk in enumerate(chunk_text(text)):
        records.append({"id": f"{source}::{i}", "text": chunk, "source": source})

print(f"Created {len(records)} chunks from {len(documents)} documents")

embedder = SentenceTransformer(EMBED_MODEL_NAME, cache_folder=str(MODEL_CACHE_DIR))
doc_texts = [r["text"] for r in records]
doc_embeddings = embedder.encode(doc_texts, show_progress_bar=True, normalize_embeddings=True)

doc_chroma_client = chromadb.PersistentClient(path="./chroma_db_wiki_docs")
doc_collection = doc_chroma_client.get_or_create_collection(
    name="source_chunks",
    metadata={"hnsw:space": "cosine"},
)

existing_ids = doc_collection.get()["ids"]
if existing_ids:
    doc_collection.delete(ids=existing_ids)

doc_collection.add(
    ids=[r["id"] for r in records],
    embeddings=[e.tolist() for e in doc_embeddings],
    documents=[r["text"] for r in records],
    metadatas=[{"source": r["source"]} for r in records],
)

print(f"Stored {doc_collection.count()} document chunks in the vector DB")


## 3. Discover candidate wiki concepts

Scan a sample of document chunks and ask Groq for the key concepts, terms, and entities in each that deserve their own wiki article. These seed the breadth-first crawl below — no hardcoded root topic, the wiki grows entirely out of what's actually in `data/`.

In [ ]:
groq_api_key = os.getenv("GROQ_API_KEY")
assert groq_api_key, "GROQ_API_KEY not found — copy .env.example to .env and add your key."
groq_client = Groq(api_key=groq_api_key)

CONCEPT_SYSTEM_PROMPT = (
    "You identify key concepts in a source document that deserve their own wiki article. "
    "Respond only with strict JSON, no prose."
)

def normalize(title):
    return re.sub(r"\s+", " ", title.strip().lower())

def slugify(title):
    return re.sub(r"[^\w]+", "-", title.lower()).strip("-") or "page"

def extract_concepts_from_chunk(chunk_text):
    prompt = f"""Text chunk:
\"\"\"
{chunk_text}
\"\"\"

List up to 5 concepts, terms, entities, or topics in this text that would deserve their own wiki article (a technology, organization, place, process, event, etc — not vague generalities).

Respond as JSON: {{"concepts": ["concept 1", "concept 2", ...]}}"""
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": CONCEPT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content).get("concepts", [])

# canonical (normalized) concept -> {"name": display name, "count": number of chunks it appeared in}
concept_counts = {}
chunks_for_concepts = records[:MAX_CONCEPT_CHUNKS]

for i, record in enumerate(chunks_for_concepts):
    try:
        concepts = extract_concepts_from_chunk(record["text"])
    except (json.JSONDecodeError, Exception) as e:
        print(f"  [skip] chunk {record['id']}: {e}")
        continue

    for concept in concepts:
        concept = str(concept).strip()
        if not concept:
            continue
        key = normalize(concept)
        if key in concept_counts:
            concept_counts[key]["count"] += 1
        else:
            concept_counts[key] = {"name": concept, "count": 1}

    if (i + 1) % 10 == 0:
        print(f"  scanned {i + 1}/{len(chunks_for_concepts)} chunks — {len(concept_counts)} unique candidate concept(s) so far")
    time.sleep(0.3)  # gentle pacing against Groq's free-tier rate limits

# most-mentioned concepts first — a simple proxy for how central they are to the document(s)
candidate_concepts = sorted(concept_counts.values(), key=lambda c: -c["count"])

print(f"\nDiscovered {len(candidate_concepts)} unique candidate concepts from {len(chunks_for_concepts)} chunks:")
for c in candidate_concepts[:20]:
    print(f"  - {c['name']} (mentioned in {c['count']} chunk(s))")


## 5. Generate the wiki (breadth-first crawl, grounded in the documents)

For each concept in the crawl queue, retrieve the most relevant document chunks. If even the closest one isn't relevant enough (cosine distance above `MIN_RELEVANCE_DISTANCE`), the concept is skipped entirely — no page is written for a concept the documents don't actually cover. Otherwise Groq writes a wiki article using **only** that retrieved context, plus a short list of related concepts to crawl next. New, not-yet-seen concepts are pushed onto the queue, breadth-first, until `MAX_PAGES` pages have been generated.


In [ ]:
WIKI_SYSTEM_PROMPT = (
    "You are a wiki author writing an encyclopedia grounded strictly in the provided source-document "
    "excerpts. Respond only with strict JSON, no prose."
)

def retrieve_doc_context(topic, k=TOP_K):
    q_embedding = embedder.encode([topic], normalize_embeddings=True)[0]
    results = doc_collection.query(query_embeddings=[q_embedding.tolist()], n_results=k)
    return [
        {"text": text, "source": meta["source"], "distance": dist}
        for text, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ]

def generate_wiki_page(topic, context_hits):
    context = "\n\n".join(f"[Source: {h['source']}]\n{h['text']}" for h in context_hits)
    prompt = f"""Source document excerpts:
{context}

Using only the excerpts above, write a wiki article about: "{topic}"

Respond as JSON:
{{
  "title": "canonical article title",
  "content": "2-4 paragraphs explaining the topic, written encyclopedia-style, grounded in the excerpts",
  "links": ["up to {MAX_LINKS_PER_PAGE} closely related topics mentioned in the excerpts that deserve their own article"]
}}"""
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": WIKI_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

pages = {}          # canonical title -> {"content": str, "links": [str, ...]}
page_sources = {}   # canonical title -> sorted list of source documents it was grounded in
visited = set()     # normalized titles already crawled (or attempted)
skipped_ungrounded = []
queue = deque(seed_topics)

while queue and len(pages) < MAX_PAGES:
    topic = queue.popleft()
    key = normalize(topic)
    if key in visited:
        continue
    visited.add(key)

    context_hits = retrieve_doc_context(topic)
    if not context_hits or context_hits[0]["distance"] > MIN_RELEVANCE_DISTANCE:
        best = f"{context_hits[0]['distance']:.3f}" if context_hits else "n/a"
        print(f"  [skip] {topic!r}: no sufficiently relevant document context (best distance={best})")
        skipped_ungrounded.append(topic)
        continue

    try:
        data = generate_wiki_page(topic, context_hits)
    except (json.JSONDecodeError, Exception) as e:
        print(f"  [skip] {topic!r}: {e}")
        continue

    title = str(data.get("title") or topic).strip()
    content = str(data.get("content", "")).strip()
    links = [str(l).strip() for l in data.get("links", [])[:MAX_LINKS_PER_PAGE] if str(l).strip()]

    pages[title] = {"content": content, "links": links}
    page_sources[title] = sorted({h["source"] for h in context_hits})
    print(f"  [{len(pages)}/{MAX_PAGES}] generated {title!r} — grounded in {len(page_sources[title])} source doc(s), {len(links)} link(s)")

    for link in links:
        if normalize(link) not in visited:
            queue.append(link)

    time.sleep(0.3)  # gentle pacing against Groq's free-tier rate limits

print(f"\nCrawled {len(pages)} pages, {len(skipped_ungrounded)} concept(s) skipped as ungrounded, "
      f"{len(queue)} link(s) left unexplored at the page budget")


## 6. Persist the wiki as Markdown

Each generated page is written to `wiki/<slug>.md` with a "See also" section and the source document(s) it was grounded in — a plain-text wiki you can browse directly on disk.


In [ ]:
WIKI_SYSTEM_PROMPT = (
    "You are a wiki author writing an encyclopedia grounded strictly in the provided source-document "
    "excerpts. Respond only with strict JSON, no prose."
)

def retrieve_doc_context(topic, k=TOP_K):
    q_embedding = embedder.encode([topic], normalize_embeddings=True)[0]
    results = doc_collection.query(query_embeddings=[q_embedding.tolist()], n_results=k)
    return [
        {"text": text, "source": meta["source"]}
        for text, meta in zip(results["documents"][0], results["metadatas"][0])
    ]

def generate_wiki_page(topic):
    context_hits = retrieve_doc_context(topic)
    context = "\n\n".join(f"[Source: {h['source']}]\n{h['text']}" for h in context_hits)
    prompt = f"""Source document excerpts:
{context}

Using only the excerpts above, write a wiki article about: "{topic}"

Respond as JSON:
{{
  "title": "canonical article title",
  "content": "2-4 paragraphs explaining the topic, written encyclopedia-style, grounded in the excerpts",
  "links": ["up to {MAX_LINKS_PER_PAGE} closely related topics mentioned in the excerpts that deserve their own article"]
}}"""
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": WIKI_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content), context_hits

pages = {}          # canonical title -> {"content": str, "links": [str, ...]}
page_sources = {}   # canonical title -> sorted list of source documents it was grounded in
visited = set()     # normalized titles already crawled (or attempted)
queue = deque(seed_topics)

while queue and len(pages) < MAX_PAGES:
    topic = queue.popleft()
    key = normalize(topic)
    if key in visited:
        continue
    visited.add(key)

    try:
        data, context_hits = generate_wiki_page(topic)
    except (json.JSONDecodeError, Exception) as e:
        print(f"  [skip] {topic!r}: {e}")
        continue

    title = str(data.get("title") or topic).strip()
    content = str(data.get("content", "")).strip()
    links = [str(l).strip() for l in data.get("links", [])[:MAX_LINKS_PER_PAGE] if str(l).strip()]

    pages[title] = {"content": content, "links": links}
    page_sources[title] = sorted({h["source"] for h in context_hits})
    print(f"  [{len(pages)}/{MAX_PAGES}] generated {title!r} — grounded in {len(page_sources[title])} source doc(s), {len(links)} link(s)")

    for link in links:
        if normalize(link) not in visited:
            queue.append(link)

    time.sleep(0.3)  # gentle pacing against Groq's free-tier rate limits

print(f"\nCrawled {len(pages)} pages, {len(queue)} link(s) left unexplored at the page budget")


## 7. Build the link graph

Nodes are wiki concepts; a directed edge `A → B` means concept `A` links to concept `B`. Linked concepts that were discovered but never crawled (because the page budget ran out) still appear as **stub nodes** — they show where the wiki would keep growing.


In [ ]:
slug_by_title = {title: slugify(title) for title in pages}

for title, data in pages.items():
    see_also = "\n".join(f"- {link}" for link in data["links"]) or "- (none)"
    sources = "\n".join(f"- {s}" for s in page_sources.get(title, [])) or "- (none)"
    page_md = f"# {title}\n\n{data['content']}\n\n## See also\n{see_also}\n\n## Source documents\n{sources}\n"
    (WIKI_DIR / f"{slug_by_title[title]}.md").write_text(page_md, encoding="utf-8")

print(f"Wrote {len(pages)} Markdown pages to {WIKI_DIR}/")
pd.DataFrame([
    {"title": t, "slug": slug_by_title[t], "links": len(d["links"]), "sources": len(page_sources.get(t, []))}
    for t, d in pages.items()
])


## 8. Visualize the wiki graph

Two views are produced:

- An **inline Plotly view** (below) — node labels are always visible; hover over a node for its status. Renders directly in the notebook output, scroll to zoom, drag to pan.
- A **standalone [pyvis](https://pyvis.readthedocs.io/) HTML file**, saved to `wiki_graph_db/` — a draggable, physics-based force layout. VS Code's notebook output sandbox doesn't run the `<script>` this needs to draw itself, so it can't be embedded inline — open the saved file directly in a browser tab for the fully interactive version.

Crawled pages are blue, stub (linked-but-not-generated) topics are grey, and (later) retrieved pages are highlighted in red.


In [ ]:
norm_to_title = {normalize(t): t for t in pages}

def resolve_link(link):
    """Map a link string to an already-crawled page's canonical title, if there is one."""
    return norm_to_title.get(normalize(link), link)

G = nx.DiGraph()
for title in pages:
    G.add_node(title, crawled=True)

for title, data in pages.items():
    for link in data["links"]:
        target = resolve_link(link)
        if target not in G:
            G.add_node(target, crawled=False)
        G.add_edge(title, target)

graphml_path = WIKI_GRAPH_DIR / "wiki_graph.graphml"
nx.write_graphml(G, graphml_path)

n_crawled = sum(1 for _, d in G.nodes(data=True) if d.get("crawled"))
n_stub = G.number_of_nodes() - n_crawled
print(f"Graph: {G.number_of_nodes()} nodes ({n_crawled} crawled, {n_stub} stub), {G.number_of_edges()} links")
print(f"Persisted to {graphml_path}")


## 9. Chunk and embed the wiki pages

Same splitter and embedding model as the document chunks above — this is a *second*, separate vector DB over the LLM-authored wiki text itself, which is what questions are actually answered from.


In [ ]:
def build_plotly_graph(graph, highlight_nodes=None, title="Wiki graph", height=650, seed=42):
    highlight_nodes = highlight_nodes or set()
    pos = nx.spring_layout(graph, seed=seed)

    edge_x, edge_y = [], []
    for u, v in graph.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(width=1, color="#666"),
        hoverinfo="none", showlegend=False,
    )

    node_x, node_y, node_text, node_hover, node_color = [], [], [], [], []
    for node, data in graph.nodes(data=True):
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(node)
        status = "crawled page" if data.get("crawled") else "stub (not yet generated)"
        node_hover.append(f"{node}<br>{status}")
        if node in highlight_nodes:
            node_color.append("#e74c3c")
        elif data.get("crawled"):
            node_color.append("#3498db")
        else:
            node_color.append("#95a5a6")

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode="markers+text",
        text=node_text, textposition="top center",
        textfont=dict(color="#e0e0e0", size=11),
        hoverinfo="text", hovertext=node_hover,
        marker=dict(size=16, color=node_color, line=dict(width=1, color="#1e1e1e")),
        showlegend=False,
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title=title,
        plot_bgcolor="#1e1e1e", paper_bgcolor="#1e1e1e",
        font=dict(color="#e0e0e0"),
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        margin=dict(l=0, r=0, t=40, b=0),
        height=height,
    )
    return fig

def export_pyvis_html(graph, path, highlight_nodes=None, height="650px"):
    highlight_nodes = highlight_nodes or set()
    net = Network(height=height, width="100%", directed=True, notebook=True,
                   cdn_resources="in_line", bgcolor="#1e1e1e", font_color="#e0e0e0")
    net.barnes_hut()

    for node, data in graph.nodes(data=True):
        status = "crawled page" if data.get("crawled") else "stub (not yet generated)"
        if node in highlight_nodes:
            color = "#e74c3c"
        elif data.get("crawled"):
            color = "#3498db"
        else:
            color = "#95a5a6"
        net.add_node(node, label=node, title=f"{node}<br>{status}", color=color)

    for u, v in graph.edges():
        net.add_edge(u, v, arrows="to")

    # pyvis's own write_html() opens the file without an explicit encoding, which crashes on
    # non-Latin1 text (e.g. Hebrew) under Windows' default locale — write it ourselves instead.
    page = net.generate_html(notebook=True)
    out_path = Path(path).resolve()
    out_path.write_text(page, encoding="utf-8")
    return out_path

full_graph_path = export_pyvis_html(G, WIKI_GRAPH_DIR / "wiki_graph_full.html")
print(f"Draggable, physics-based view saved to: {full_graph_path}\n(open it directly in a browser tab)")

build_plotly_graph(G, title="Full wiki graph").show()


## 10. Ask a question — retrieve chunks and highlight source pages

Embed the question, run a similarity search against the wiki vector DB, then highlight the retrieved pages (and their direct links, out to `HOPS` hops) in the wiki graph so you can see where the answer is coming from.


In [ ]:
wiki_records = []  # each: {id, text, source}
for title, data in pages.items():
    for i, chunk in enumerate(chunk_text(data["content"])):
        wiki_records.append({"id": f"{title}::{i}", "text": chunk, "source": title})

print(f"Created {len(wiki_records)} chunks from {len(pages)} wiki pages")

wiki_texts = [r["text"] for r in wiki_records]
wiki_embeddings = embedder.encode(wiki_texts, show_progress_bar=True, normalize_embeddings=True)

wiki_chroma_client = chromadb.PersistentClient(path="./chroma_db_wiki_pages")
wiki_collection = wiki_chroma_client.get_or_create_collection(
    name="wiki_chunks",
    metadata={"hnsw:space": "cosine"},
)

existing_ids = wiki_collection.get()["ids"]
if existing_ids:
    wiki_collection.delete(ids=existing_ids)

wiki_collection.add(
    ids=[r["id"] for r in wiki_records],
    embeddings=[e.tolist() for e in wiki_embeddings],
    documents=[r["text"] for r in wiki_records],
    metadatas=[{"source": r["source"]} for r in wiki_records],
)

print(f"Stored {wiki_collection.count()} wiki chunks in the vector DB")


## 11. Compile an answer with Groq (`llama-3.3-70b-versatile`)

The retrieved chunks are stitched into a context block and sent to the LLM along with the question — the same pattern as `rag_demo.ipynb`. The answer is also rendered as right-to-left HTML directly in an output cell below (via an embedded iframe) and saved to `answer.html`, matching the other notebooks — since Hebrew text renders far better that way than in a plain text output.


In [ ]:
def retrieve(question, k=TOP_K):
    q_embedding = embedder.encode([question], normalize_embeddings=True)[0]
    results = wiki_collection.query(query_embeddings=[q_embedding.tolist()], n_results=k)
    hits = []
    for text, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        hits.append({"text": text, "source": meta["source"], "distance": dist})
    return hits, q_embedding

def neighborhood_subgraph(seed_pages, hops=HOPS):
    nodes_in_subgraph = set(seed_pages)
    frontier = set(seed_pages)
    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            if node in G:
                next_frontier.update(G.predecessors(node))
                next_frontier.update(G.successors(node))
        nodes_in_subgraph.update(next_frontier)
        frontier = next_frontier
    return G.subgraph(nodes_in_subgraph).copy()

question = "לאילו מדינות ישראל מייצאת גז ואילו כמויות?"
hits, q_embedding = retrieve(question)

for h in hits:
    print(f"[{h['source']}] (distance={h['distance']:.3f})\n{h['text']}\n")

seed_pages = {h["source"] for h in hits}
subgraph = neighborhood_subgraph(seed_pages)

subgraph_path = export_pyvis_html(subgraph, WIKI_GRAPH_DIR / "wiki_graph_subgraph.html", highlight_nodes=seed_pages, height="500px")
print(f"Draggable, physics-based view saved to: {subgraph_path}\n(open it directly in a browser tab)")

build_plotly_graph(subgraph, highlight_nodes=seed_pages, title=f'Retrieved pages for: "{question}"', height=500).show()


## 12. End-to-end `ask()` helper

Combine retrieval + graph highlighting + generation into a single function for interactive use.


In [ ]:
def compile_answer(question, hits):
    context = "\n\n".join(f"[Source: {h['source']}]\n{h['text']}" for h in hits)
    prompt = f"""Answer the question using only the context below. If the context doesn't contain the answer, say so.

Context:
{context}

Question: {question}
Answer:"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers strictly from the provided context and cites the source page(s) you used."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

def render_answer_html(question, answer, path="answer.html", height=420):
    page = f"""<!DOCTYPE html>
<html lang="he" dir="rtl">
<head>
<meta charset="utf-8">
<title>LLM Wiki Answer</title>
<style>
    body {{
        font-family: "Segoe UI", Arial, sans-serif;
        margin: 20px;
        line-height: 1.7;
        background: #1e1e1e;
        color: #e0e0e0;
    }}
    h2 {{ font-size: 1rem; color: #999; margin-bottom: 4px; }}
    .question {{ font-size: 1.3rem; font-weight: bold; margin-bottom: 24px; color: #f5f5f5; }}
    .answer {{ font-size: 1.15rem; white-space: pre-wrap; color: #e0e0e0; }}
</style>
</head>
<body>
    <h2>שאלה</h2>
    <div class="question">{html_lib.escape(question)}</div>
    <h2>תשובה</h2>
    <div class="answer">{html_lib.escape(answer)}</div>
</body>
</html>
"""
    out_path = Path(path).resolve()
    out_path.write_text(page, encoding="utf-8")

    iframe = (
        f'<iframe srcdoc="{html_lib.escape(page)}" '
        f'style="width:100%; height:{height}px; border:1px solid #444; border-radius:6px;">'
        f"</iframe>"
    )
    display(HTML(iframe))
    return out_path

answer = compile_answer(question, hits)
print(answer)

html_path = render_answer_html(question, answer)


## Next steps

- Raise `MAX_CONCEPT_CHUNKS`/`MAX_SEED_CONCEPTS`/`MAX_PAGES` to scan more of the source documents and grow the wiki further (slower, more Groq calls — watch your rate limits)
- Point `DATA_DIR` at a different folder to build a wiki about a completely different set of documents
- Tune `MIN_RELEVANCE_DISTANCE` — lower it to be stricter about what counts as "grounded", or raise it if too many legitimate concepts are getting skipped
- Improve link resolution — right now merging is exact normalized-string match, so near-duplicate concept names become separate nodes; try fuzzy matching or an LLM-based dedup pass
- Try `HOPS = 2` for broader highlighted context, or prune low-connectivity stub nodes before visualizing
- Reload the wiki from `wiki/*.md` and the graph from `wiki_graph_db/wiki_graph.graphml` on startup instead of re-crawling every run
- Swap `nx.spring_layout(...)` for `nx.kamada_kawai_layout(...)` for a different inline layout, or tune `net.set_options(...)` (pyvis) for the browser-viewed file


In [ ]:
def ask(question, k=TOP_K, hops=HOPS, verbose=True):
    hits, _ = retrieve(question, k=k)
    seed_pages = {h["source"] for h in hits}
    subgraph = neighborhood_subgraph(seed_pages, hops=hops)
    answer = compile_answer(question, hits)

    render_answer_html(question, answer)

    graph_path = export_pyvis_html(subgraph, WIKI_GRAPH_DIR / "wiki_graph_last_query.html", highlight_nodes=seed_pages, height="450px")
    print(f"Draggable, physics-based view saved to: {graph_path}\n(open it directly in a browser tab)")
    build_plotly_graph(subgraph, highlight_nodes=seed_pages, title=f'Retrieved pages for: "{question}"', height=450).show()

    if verbose:
        print("Retrieved chunks:")
        for h in hits:
            print(f"  - [{h['source']}] {h['text'][:100]}...")
        print("\nAnswer:\n" + answer)
    return answer

# Try it yourself:
ask("לאילו מדינות ישראל מייצאת גז ואילו כמויות?")


## Next steps

- Raise `MAX_CONCEPT_CHUNKS`/`MAX_PAGES` to scan more of the source documents and grow the wiki further (slower, more Groq calls — watch your rate limits)
- Point `DATA_DIR` at a different folder to build a wiki about a completely different set of documents
- Improve link resolution — right now merging is exact normalized-string match, so near-duplicate concept names become separate nodes; try fuzzy matching or an LLM-based dedup pass
- Tighten or loosen the grounding prompt in `generate_wiki_page` — e.g. require a minimum retrieval relevance before writing a page, or allow a bit of general knowledge to fill gaps
- Try `HOPS = 2` for broader highlighted context, or prune low-connectivity stub nodes before visualizing
- Reload the wiki from `wiki/*.md` and the graph from `wiki_graph_db/wiki_graph.graphml` on startup instead of re-crawling every run
- Swap `nx.spring_layout(...)` for `nx.kamada_kawai_layout(...)` for a different inline layout, or tune `net.set_options(...)` (pyvis) for the browser-viewed file